Thie notebook creates simulations to contextualize the FN and FP rate for the hypothesis "cre of interest is significantly different from minP".
- negative simulations (cre_oi=minP) for quantifying the FP rate.
- positive simulations (cre_oi!=minP) for quantifying FN rate.

# Setup

In [1]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import dask.dataframe as dd
 

%load_ext autoreload
%autoreload 2

2025-11-19 15:55:48.061419: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-19 15:55:48.120410: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='32GB')
client=Client(cluster)

In [3]:
data_root="/home/mcn26/project_pi_skr2/shared/tabula_data"

# Creating an artificial library

In [4]:
#making up the CREs
spread_gt,spread_hypothesis=scm.activity_spread(
    cell_types=list(scm.SHENDURE_BOUNDS.cells_per_cell_type.keys()),
    minimum=scm.SHENDURE_BOUNDS.min_mpra_umi,
    maximum=scm.SHENDURE_BOUNDS.max_mpra_umi,
    minp_value=scm.SHENDURE_BOUNDS.reference_activity,
    total=100,
    frac_active=0.5,
    ct_specificity=.2)


library=scm.simulate_library(CREs=spread_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)

scMPRAforge: INFO: 82.0% of active elements are not cell-type specific.


In [5]:
library.to_csv(f"{data_root}/simulated/shendure_pow_analysis/library_20251119.tsv",sep="\t")

In [6]:
batch=scm.de_novo_simulation(
                        simulation_replicates=5,
                        experiment_bounds=scm.SHENDURE_BOUNDS,
                        ground_truth=spread_gt,
                        library=library)
batch.gamut(client)

scMPRAforge: WARNING: 7523/256603 cells (2.932%) have ≥1 multi-transfection event.
scMPRAforge: WARNING: Multi-transfections exceed threshold of 2.000%!
scMPRAforge: WARNING: 7830/256692 cells (3.050%) have ≥1 multi-transfection event.
scMPRAforge: WARNING: Multi-transfections exceed threshold of 2.000%!


In [8]:
batch.save(f"{data_root}/simulated/shendure_pow_analysis","sim_20251119")

In [ ]:
spread_hypothesis.to_tsv(f"{data_root}/simulated/shendure_pow_analysis/spread_hypothesis_20251119.tsv")

In [13]:
spread_gt.to_csv(f"{data_root}/simulated/shendure_pow_analysis/spread_gt_20251119.tsv",sep="\t")

In [9]:
client.close()
cluster.close()